# 01 — Quickstart: Your First Optimization with qanneal

**What you'll learn in this notebook:**
- What QUBO and Ising problems are (with concrete examples)
- How to encode a real combinatorial problem
- Solve with Simulated Annealing (SA) and Simulated Quantum Annealing (SQA)
- Read and interpret results
- Visualise energy convergence

> **Time**: ~10 minutes to read + run

In [ ]:
# ── Path setup ────────────────────────────────────────────────────────────────
# Only needed when running notebooks from the repo without a system install.
import os, sys
_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
_PY   = os.path.join(_ROOT, 'python')
if _PY not in sys.path:
    sys.path.insert(0, _PY)

import numpy as np
import matplotlib.pyplot as plt
import qanneal
print('qanneal', qanneal.__version__)

---

## 1 — What is a QUBO problem?

Many NP-hard problems can be written as:

$$\min_{x \in \{0,1\}^n} \; \sum_i \sum_j Q_{ij}\, x_i x_j$$

where $x_i \in \{0, 1\}$ are binary decision variables.

- **Diagonal** entries $Q_{ii}$: linear (bias) term for variable $i$.
- **Off-diagonal** entries $Q_{ij}$ ($i \neq j$): coupling between $i$ and $j$.

**Toy example**: maximise $x_0 + x_1$ subject to `not (x_0 AND x_1)` (at most one can be 1).

We can write this as a QUBO with penalty:

$$\min \; -x_0 - x_1 + 3\,x_0 x_1$$

The penalty term 3x₀x₁ makes choosing both x₀=x₁=1 very expensive.

In [ ]:
from qanneal import QUBO

# Q[i,i] = linear bias, Q[i,j] = coupling
# Minimise: -x0 - x1 + 3*x0*x1
Q = np.array([
    [-1.0,  3.0],   # Q[0,0]=-1 (bias for x0), Q[0,1]=3 (coupling)
    [ 0.0, -1.0],   # Q[1,0]=0, Q[1,1]=-1 (bias for x1)
], dtype=float)

qubo = QUBO(Q)

# Enumerate all 4 solutions
print('x0  x1  QUBO energy')
for x0 in [0, 1]:
    for x1 in [0, 1]:
        x = np.array([x0, x1])
        E = x @ Q @ x
        marker = '  ← best' if E == min(x @ Q @ x for x in [np.array([a,b]) for a in [0,1] for b in [0,1]]) else ''
        print(f' {x0}   {x1}   {E:+.1f}{marker}')

In [ ]:
# Solve with qanneal — use return_bits=True to get {0,1} back
from qanneal import solve

result = solve(Q, method='sa', reads=10, seed=0, return_bits=True)
print('Best solution:', result.best_sample)   # should be [1, 0] or [0, 1]
print('Best energy:  ', result.best_energy)    # should be -1.0

---

## 2 — What is an Ising problem?

The **Ising model** uses spins $s_i \in \{-1, +1\}$ instead of bits:

$$E(s) = \sum_i h_i s_i + \sum_{i < j} J_{ij}\, s_i s_j + c$$

- $h_i$: local field (positive → spin prefers −1; negative → spin prefers +1)
- $J_{ij}$: coupling (negative = ferromagnetic, aligned spins preferred; positive = antiferromagnetic)
- $c$: constant (irrelevant for optimization)

QUBO and Ising are **equivalent** via $x = (s+1)/2$. qanneal converts automatically.

---

## 3 — A Real Problem: Number Partition

**Problem**: Given numbers $a_1, \ldots, a_n$, assign each a sign $s_i \in \{-1, +1\}$ to minimise $|\sum_i a_i s_i|$.

This is equivalent to splitting the list into two groups with equal (or nearly equal) sums. It is **NP-hard** for large instances.

**Ising encoding**:
$$\left(\sum_i a_i s_i\right)^2 = \sum_i a_i^2 + 2\sum_{i<j} a_i a_j\, s_i s_j$$

So: $h_i = 0$,  $J_{ij} = 2 a_i a_j$,  $c = \sum_i a_i^2$

In [ ]:
from qanneal import DenseIsing

# Numbers to partition
nums = np.array([3, 5, 7, 9, 11, 13, 15, 17], dtype=float)
n = len(nums)
print(f'Sum = {nums.sum():.0f}  →  target half-sum = {nums.sum()/2:.1f}')

# Build Ising
h = np.zeros(n)
J = np.zeros((n, n))
for i in range(n):
    for j in range(i+1, n):
        J[i, j] = J[j, i] = 2.0 * nums[i] * nums[j]
c = float(np.dot(nums, nums))

ising = DenseIsing(h, J, c=c)

# Quick sanity check: manually compute E for all-+1 solution
all_plus = [1]*n
E_all_plus = ising.energy(all_plus)
diff_all_plus = abs(nums.sum())  # all in group A
print(f'All +1: E={E_all_plus:.0f},  diff={diff_all_plus:.0f}  (expected: {diff_all_plus**2:.0f} from Ising encoding)')

### 3.1 — Brute force (for reference, only feasible for small n)

In [ ]:
best_diff = float('inf')
best_spins = None
for mask in range(1 << n):
    spins = np.array([1 if (mask >> i) & 1 else -1 for i in range(n)])
    diff = abs(float(np.dot(nums, spins)))
    if diff < best_diff:
        best_diff = diff
        best_spins = spins

print(f'Brute-force optimum: diff = {best_diff}')
print(f'Spins: {best_spins}')
group_a = nums[best_spins == +1]
group_b = nums[best_spins == -1]
print(f'Group A: {group_a.astype(int).tolist()}  (sum={group_a.sum():.0f})')
print(f'Group B: {group_b.astype(int).tolist()}  (sum={group_b.sum():.0f})')

### 3.2 — Solve with SA

In [ ]:
from qanneal import auto_schedule_sa_tuned

schedule = auto_schedule_sa_tuned(ising, mode='balanced')
print(f'SA schedule: {len(schedule.betas)} steps,  β = [{schedule.betas[0]:.3f} → {schedule.betas[-1]:.3f}]')

sa_result = solve(
    ising,
    method='sa',
    reads=30,
    sweeps_per_beta=40,
    schedule=schedule,
    seed=42,
    progress=False,
)

sa_spins = sa_result.best_sample
sa_diff  = abs(float(np.dot(nums, sa_spins)))
print(f'SA best diff: {sa_diff}  (optimal: {best_diff})')
print(f'SA best energy: {sa_result.best_energy:.2f}')

### 3.3 — Solve with SQA

In [ ]:
from qanneal import auto_schedule_sqa_tuned

schedule_sqa = auto_schedule_sqa_tuned(ising, mode='balanced')
print(f'SQA schedule: {len(schedule_sqa.betas)} steps')
print(f'  β = [{schedule_sqa.betas[0]:.3f} → {schedule_sqa.betas[-1]:.3f}]')
print(f'  Γ = [{schedule_sqa.gammas[0]:.3f} → {schedule_sqa.gammas[-1]:.4f}]  (geometric decay)')

sqa_result = solve(
    ising,
    method='sqa',
    reads=30,
    sweeps_per_beta=40,
    worldline_sweeps=4,
    trotter_slices=16,
    schedule=schedule_sqa,
    seed=42,
    progress=False,
)

sqa_spins = sqa_result.best_sample
sqa_diff  = abs(float(np.dot(nums, sqa_spins)))
print(f'SQA best diff: {sqa_diff}  (optimal: {best_diff})')

### 3.4 — Compare energy distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Energy histograms
ax = axes[0]
bins = np.linspace(
    min(min(sa_result.energies), min(sqa_result.energies)) - 10,
    max(max(sa_result.energies), max(sqa_result.energies)) + 10,
    20
)
ax.hist(sa_result.energies,  bins=bins, alpha=0.7, label='SA',  color='#4c8bf5')
ax.hist(sqa_result.energies, bins=bins, alpha=0.7, label='SQA', color='#c84b31')
ax.axvline(best_diff**2 + c, color='green', lw=2, linestyle='--', label=f'Optimal E={best_diff**2+c:.0f}')
ax.set_xlabel('Energy'); ax.set_ylabel('Count')
ax.set_title('Energy distribution (30 reads each)')
ax.legend()

# Energy trace (first read)
ax2 = axes[1]
ax2.plot(sa_result.trace,  label='SA',  color='#4c8bf5', lw=1.5)
ax2.plot(sqa_result.trace, label='SQA', color='#c84b31', lw=1.5)
ax2.set_xlabel('Temperature step'); ax2.set_ylabel('Energy')
ax2.set_title('Energy vs annealing step (first read)')
ax2.legend()

fig.suptitle(f'Number partition: n={n}  sum={nums.sum():.0f}  optimal_diff={best_diff}', fontsize=12)
fig.tight_layout()
plt.show()

---

## 4 — SQAPT: The Best Quantum Analog

In [ ]:
from qanneal import auto_ladder_sqa_tuned

ladder = auto_ladder_sqa_tuned(ising, replicas=8, mode='balanced')
print(f'SQAPT ladder: {len(ladder.betas)} rungs')
for i, (b, g) in enumerate(zip(ladder.betas, ladder.gammas)):
    print(f'  rung {i}: β={b:.3f}  Γ={g:.3f}')

sqapt_result = solve(
    ising,
    method='sqapt',
    reads=20,
    sweeps_per_beta=40,
    worldline_sweeps=4,
    trotter_slices=16,
    replicas=8,
    pt_steps=60,
    schedule=ladder,
    seed=42,
    progress=False,
)

sqapt_diff = abs(float(np.dot(nums, sqapt_result.best_sample)))
print(f'\nSQAPT best diff: {sqapt_diff}  (optimal: {best_diff})')

In [ ]:
# Summary table
print('Method     best_diff   mean_energy   min_energy')
print('-' * 50)
for name, res, diff in [('SA',    sa_result,    sa_diff),
                         ('SQA',   sqa_result,   sqa_diff),
                         ('SQAPT', sqapt_result, sqapt_diff)]:
    mean_e = np.mean(res.energies)
    min_e  = np.min(res.energies)
    print(f'{name:<10} {diff:<12.1f} {mean_e:<14.1f} {min_e:.1f}')
print(f"{'Optimal':<10} {best_diff:<12.1f}")

---

## 5 — Key Takeaways

| Concept | Meaning |
|---------|--------|
| **QUBO** | Minimise quadratic energy over bits {0,1} — common encoding for combinatorial problems |
| **Ising** | Same problem with spins {−1,+1} — native format for all annealers |
| **reads** | Multiple independent runs; take the best |
| **sweeps_per_beta** | More sweeps per temperature = better equilibration |
| **SA** | Good baseline; fast; misses narrow barriers |
| **SQA** | Adds quantum tunneling via imaginary-time Trotter slices |
| **SQAPT** | SA+PT replicas explore the (β, Γ) landscape simultaneously — best general-purpose |

**Next**: `02_sqa_physics.ipynb` — deep dive into how SQA works.